# DINOv2 + MLP: AI-Generated Image Detection
Uses a frozen `dinov2_vits14` backbone to extract patch-level embeddings, then trains a lightweight MLP head for binary classification (real vs AI-generated).

## 1. Setup

In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Config

In [2]:
CFG = {
    # ── Paths ────────────────────────────────────────────────────────────────────────────
    "dataset_path": "/workspace/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0",
    "ckpt_dir":     os.path.dirname(os.path.abspath("dinov2_mlp_best.pth")),

    # ── Training ──────────────────────────────────────────────────────────────────
    "batch_size":     64,
    "val_batch_size": 128,       # no gradients → can afford 2× batch
    "epochs":         10,
    "lr":             3e-4,
    "weight_decay":   1e-4,
    "num_workers":    8,

    # ── Image ──────────────────────────────────────────────────────────────────────
    "img_size":       224,       # DINOv2 ViT-S/14 expects multiples of 14
    "val_split":      0.1,
    "seed":           SEED,
}

print("Config:")
for k, v in CFG.items():
    print(f"  {k}: {v}")

Config:
  dataset_path: /workspace/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0
  ckpt_dir: /workspace
  batch_size: 64
  val_batch_size: 128
  epochs: 10
  lr: 0.0003
  weight_decay: 0.0001
  num_workers: 8
  img_size: 224
  val_split: 0.1
  seed: 42


## 3. Transforms
Training uses `RandomResizedCrop` + `HorizontalFlip` for data augmentation.  
Validation uses a standard `Resize(256)` → `CenterCrop(224)` pipeline — identical to the ImageNet evaluation protocol used when pre-training DINOv2.

In [3]:
IMG_SIZE = CFG["img_size"]  # 224

# ImageNet statistics — DINOv2 was pre-trained with these
_MEAN = (0.485, 0.456, 0.406)
_STD  = (0.229, 0.224, 0.225)

# albumentations does not have CenterCrop+Resize as a single op for val,
# so we use torchvision for the validation pipeline.
train_transform = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.08, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=_MEAN, std=_STD),
    ToTensorV2(),
])

val_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=_MEAN, std=_STD),
])

print("Train transform (albumentations):")
for t in train_transform.transforms:
    print(f"  {t.__class__.__name__}")

print("\nVal transform (torchvision):")
for t in val_transform.transforms:
    print(f"  {t.__class__.__name__}")


Train transform (albumentations):
  RandomResizedCrop
  HorizontalFlip
  Normalize
  ToTensorV2

Val transform (torchvision):
  Resize
  CenterCrop
  ToTensor
  Normalize


## 4. Dataset
Reads `labels.csv` from the shard directory and loads RGB images on demand.  
Each sample returns `(image_tensor, label)` where `label ∈ {0, 1}` (0 = real, 1 = AI-generated).

In [4]:
class ImageDataset(Dataset):
    """
    Minimal image dataset for binary classification.

    Expected shard layout
    ---------------------
    shard_root/
        labels.csv          # columns: image_name, label
        images/
            <image_name>    # any PIL-readable format

    Parameters
    ----------
    shard_path : str
        Root directory of the shard.
    indices : list[int] | None
        Optional subset of row indices. If None, uses all rows.
    transform : callable | None
        Applied to the PIL image. May be an albumentations Compose or
        a torchvision Compose — detected automatically.
    """

    def __init__(self, shard_path: str, indices=None, transform=None):
        self.shard_path = shard_path
        self.transform  = transform
        self.labels     = pd.read_csv(os.path.join(shard_path, "labels.csv"))
        if indices is not None:
            self.labels = self.labels.iloc[indices].reset_index(drop=True)
        print(f"[ImageDataset] {len(self.labels)} samples from {shard_path}")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        row      = self.labels.iloc[idx]
        img_path = os.path.join(self.shard_path, "images", row["image_name"])
        pil_img  = Image.open(img_path).convert("RGB")

        if isinstance(self.transform, A.Compose):
            # albumentations expects a numpy array
            img_tensor = self.transform(image=np.array(pil_img))["image"]
        elif self.transform is not None:
            # torchvision / other callable, operates on PIL
            img_tensor = self.transform(pil_img)
        else:
            img_tensor = transforms.ToTensor()(pil_img)

        label = torch.tensor(row["label"], dtype=torch.long)
        return img_tensor, label


# ── Build train / val splits ──────────────────────────────────────────────────
if os.path.exists(CFG["dataset_path"]):
    _all_labels = pd.read_csv(os.path.join(CFG["dataset_path"], "labels.csv"))
    all_indices = list(range(len(_all_labels)))

    train_idx, val_idx = train_test_split(
        all_indices,
        test_size=CFG["val_split"],
        random_state=CFG["seed"],
        stratify=_all_labels["label"],
    )
    print(f"Train samples : {len(train_idx)} | Val samples : {len(val_idx)}")

    train_dataset = ImageDataset(CFG["dataset_path"], indices=train_idx, transform=train_transform)
    val_dataset   = ImageDataset(CFG["dataset_path"], indices=val_idx,   transform=val_transform)

    # Smoke-test a single sample
    _img, _lbl = train_dataset[0]
    print(f"Sample tensor shape : {tuple(_img.shape)}")
    print(f"Sample label        : {_lbl.item()}")
else:
    print(f"[INFO] Dataset path not found locally — skipping dataset build.")
    train_dataset = val_dataset = None


Train samples : 45000 | Val samples : 5000
[ImageDataset] 45000 samples from /workspace/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0
[ImageDataset] 5000 samples from /workspace/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0
Sample tensor shape : (3, 224, 224)
Sample label        : 1


## 6. DataLoaders
`pin_memory=True` enables async DMA transfers so CPU→GPU copies overlap with the forward pass — a free win on GPU machines.  
`persistent_workers=True` keeps the multiprocessing pool alive between epochs, avoiding per-epoch worker-spawn overhead.


In [5]:
if train_dataset is not None:
    _loader_kwargs = dict(
        num_workers=CFG["num_workers"],
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG["batch_size"],
        shuffle=True,
        **_loader_kwargs,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CFG["val_batch_size"],
        shuffle=False,
        **_loader_kwargs,
    )

    print(f"Train batches : {len(train_loader)} | Val batches : {len(val_loader)}")
else:
    print("[INFO] Datasets not available — skipping DataLoader creation.")
    train_loader = val_loader = None


Train batches : 704 | Val batches : 40


## 5. Load DINOv2 Backbone
`dinov2_vits14` is ViT-S/14 pre-trained with self-supervised DINO v2 on LVD-142M.  
All backbone parameters are **frozen** — only the MLP head (added later) will be trained.  
The embedding dimension is inferred from a dummy forward pass so the code adapts automatically if a different variant is swapped in.

In [6]:
# Load pre-trained DINOv2 ViT-S/14 from torch.hub
backbone = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14",
    pretrained=True,
)
backbone = backbone.to(device)

# Freeze all backbone parameters
for param in backbone.parameters():
    param.requires_grad = False
backbone.eval()

# Infer the CLS-token embedding dimension via a dummy forward pass
with torch.no_grad():
    _dummy = torch.zeros(1, 3, CFG["img_size"], CFG["img_size"], device=device)
    _out   = backbone(_dummy)          # DINOv2 hub models return the CLS token by default
    EMBED_DIM = _out.shape[-1]
    del _dummy, _out

total_params     = sum(p.numel() for p in backbone.parameters())
trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)

print(f"Backbone         : dinov2_vits14")
print(f"Embedding dim    : {EMBED_DIM}")
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}  (all frozen)")

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Backbone         : dinov2_vits14
Embedding dim    : 384
Total params     : 22,056,576
Trainable params : 0  (all frozen)


## 7. Embedding Wrapper
Thin `nn.Module` that forwards images through the frozen DINOv2 backbone and returns the CLS-token vector.  
Wrapping the backbone in a module keeps gradient suppression (`@torch.no_grad()`) self-contained and makes it easy to compose with the classifier head.


In [7]:
class DINOv2Embedder(nn.Module):
    """
    Frozen DINOv2 backbone wrapper.

    Accepts a batch of images (B, 3, H, W) and returns the CLS-token
    embedding (B, EMBED_DIM).  The backbone is always run without gradients.
    """

    def __init__(self, backbone: nn.Module):
        super().__init__()
        self.backbone = backbone

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)   # (B, EMBED_DIM)


# Smoke-test
embedder = DINOv2Embedder(backbone).to(device)
with torch.no_grad():
    _probe = torch.zeros(2, 3, CFG["img_size"], CFG["img_size"], device=device)
    _emb   = embedder(_probe)
    print(f"Embedder output shape : {tuple(_emb.shape)}")   # (2, 384) for ViT-S/14
    del _probe, _emb


Embedder output shape : (2, 384)


## 8. MLP Classifier
Lightweight two-layer MLP head: `EMBED_DIM → 256 → 1`.  
A single logit is used with `BCEWithLogitsLoss` for numerically stable binary cross-entropy.  
`Dropout(0.3)` before the first linear layer regularises the head during training.


In [8]:
class MLPClassifier(nn.Module):
    """
    Two-layer MLP head: embed_dim → 256 → 1.

    Parameters
    ----------
    embed_dim : int   — input dimension (from DINOv2 backbone)
    dropout   : float — dropout probability applied before the first layer
    """

    def __init__(self, embed_dim: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(embed_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1),    # single logit → BCEWithLogitsLoss
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : (B, embed_dim)

        Returns
        -------
        logits : (B, 1)
        """
        return self.net(x)


# Smoke-test
_clf = MLPClassifier(EMBED_DIM, dropout=0.3)
_x   = torch.zeros(4, EMBED_DIM)
with torch.no_grad():
    _out = _clf(_x)
print(f"MLPClassifier output shape : {tuple(_out.shape)}")   # (4, 1)
del _clf, _x, _out


MLPClassifier output shape : (4, 1)


## 9. Full Model
`DINOv2Classifier` composes the frozen embedder and the trainable MLP head into a single module.  
Only the MLP parameters have `requires_grad=True`, so the optimizer only receives those when passed `model.parameters()` — but receiving all parameters and relying on `requires_grad=False` for the backbone also works correctly.


In [9]:
class DINOv2Classifier(nn.Module):
    """
    End-to-end model: image → DINOv2 backbone (frozen) → MLP head → logit.

    Only the MLP head is trained; the backbone always runs without gradients.
    """

    def __init__(self, backbone: nn.Module, embed_dim: int, dropout: float = 0.3):
        super().__init__()
        self.embedder   = DINOv2Embedder(backbone)
        self.classifier = MLPClassifier(embed_dim, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : (B, 3, H, W)

        Returns
        -------
        logits : (B, 1)
        """
        embeddings = self.embedder(x)       # (B, EMBED_DIM) — no grad
        return self.classifier(embeddings)  # (B, 1)


# ── Instantiate and move to device ────────────────────────────────────────────
model = DINOv2Classifier(backbone, embed_dim=EMBED_DIM, dropout=0.3).to(device)

# ── Parameter summary ─────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f"Model        : DINOv2Classifier (dinov2_vits14 + MLP)")
print(f"Total params : {total_params:,}")
print(f"Trainable    : {trainable_params:,}  (MLP head only)")
print(f"Frozen       : {frozen_params:,}  (DINOv2 backbone)")

# ── Forward-pass shape check ──────────────────────────────────────────────────
with torch.no_grad():
    _probe  = torch.zeros(2, 3, CFG["img_size"], CFG["img_size"], device=device)
    _logits = model(_probe)
    print(f"Output shape (batch=2): {tuple(_logits.shape)}")   # (2, 1)
    del _probe, _logits


Model        : DINOv2Classifier (dinov2_vits14 + MLP)
Total params : 22,155,393
Trainable    : 98,817  (MLP head only)
Frozen       : 22,056,576  (DINOv2 backbone)


Output shape (batch=2): (2, 1)


## 10. Training Setup
AdamW optimizer with cosine-annealed learning rate. Only the trainable MLP head parameters are passed so the frozen backbone is never updated.


In [10]:
# Only pass trainable parameters to the optimizer
trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)

# Cosine decay from lr → eta_min over all training epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"],
    eta_min=1e-6,
)

criterion = nn.BCEWithLogitsLoss()

# AMP scaler — disabled automatically when CUDA is unavailable
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print(f"Optimizer : AdamW  (lr={CFG['lr']}, weight_decay={CFG['weight_decay']})")
print(f"Scheduler : CosineAnnealingLR  (T_max={CFG['epochs']}, eta_min=1e-6)")
print(f"Loss      : BCEWithLogitsLoss")
print(f"AMP scaler enabled: {scaler.is_enabled()}")
print(f"Params handed to optimizer: {sum(p.numel() for p in trainable_params):,}")


Optimizer : AdamW  (lr=0.0003, weight_decay=0.0001)
Scheduler : CosineAnnealingLR  (T_max=10, eta_min=1e-6)
Loss      : BCEWithLogitsLoss
AMP scaler enabled: True
Params handed to optimizer: 98,817


## 11. Training & Validation Functions
`train_one_epoch` runs a full pass over the training set with AMP + GradScaler.  
`validate` runs inference on a loader and returns `(accuracy, roc_auc)`.


In [11]:
def train_one_epoch(loader, log_interval: int = 50) -> float:
    """
    One full pass over the training set with AMP + GradScaler.

    Returns
    -------
    mean loss over all batches
    """
    model.train()
    # Keep backbone in eval mode so BN statistics stay frozen
    model.embedder.backbone.eval()
    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(tqdm(loader, desc="  train", leave=False)):
        images = images.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
            logits = model(images)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        if (batch_idx + 1) % log_interval == 0:
            avg = running_loss / (batch_idx + 1)
            tqdm.write(f"    batch {batch_idx + 1:4d}/{len(loader)} | loss {avg:.5f}")

    return running_loss / len(loader)


@torch.no_grad()
def validate(loader) -> tuple[float, float]:
    """
    Evaluate on a loader, returning (accuracy, roc_auc).

    Returns
    -------
    accuracy : fraction of correct binary predictions (threshold = 0.5)
    roc_auc  : area under the ROC curve
    """
    model.eval()
    all_probs  = []
    all_labels = []

    for images, labels in tqdm(loader, desc="  val  ", leave=False):
        images = images.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
            logits = model(images)

        probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()   # (B,)
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.numpy().tolist())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    preds    = (all_probs >= 0.5).astype(int)
    accuracy = (preds == all_labels).mean()
    roc_auc  = roc_auc_score(all_labels, all_probs)

    return float(accuracy), float(roc_auc)

print("train_one_epoch and validate defined.")


train_one_epoch and validate defined.


## 12. Epoch Training
Train for `CFG["epochs"]` epochs. Best checkpoint is saved whenever validation AUC improves.


In [12]:
BEST_PATH  = os.path.join(CFG["ckpt_dir"], "dinov2_mlp_best.pth")
FINAL_PATH = os.path.join(CFG["ckpt_dir"], "dinov2_mlp_final.pth")

best_auc      = 0.0
history: list[dict] = []

print("=" * 65)
print(f"  DINOv2 MLP training  —  {CFG['epochs']} epochs  |  device: {device}")
print("=" * 65)

for epoch in range(1, CFG["epochs"] + 1):
    print(f"\nEpoch {epoch:2d}/{CFG['epochs']}")

    train_loss       = train_one_epoch(train_loader)
    val_acc, val_auc = validate(val_loader)
    current_lr       = optimizer.param_groups[0]["lr"]

    print(f"  train loss : {train_loss:.5f}")
    print(f"  val acc    : {val_acc * 100:.2f}%")
    print(f"  val AUC    : {val_auc:.5f}")
    print(f"  lr         : {current_lr:.2e}")

    history.append({
        "epoch":      epoch,
        "train_loss": train_loss,
        "val_acc":    val_acc,
        "val_auc":    val_auc,
        "lr":         current_lr,
    })

    # ── Checkpoint: save best ─────────────────────────────────────────────
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  ✓ New best AUC {best_auc:.5f} — saved to {BEST_PATH}")

    scheduler.step()
    torch.cuda.empty_cache()

print("\n" + "=" * 65)
print(f"Training complete.  Best val AUC: {best_auc:.5f}")
print("=" * 65)

# ── Training history ──────────────────────────────────────────────────────────
history_df = pd.DataFrame(history)
history_csv = os.path.join(CFG["ckpt_dir"], "dinov2_mlp_training_history.csv")
history_df.to_csv(history_csv, index=False)
print(f"\nTraining history saved to: {history_csv}")
print(history_df.to_string(index=False))


  DINOv2 MLP training  —  10 epochs  |  device: cuda

Epoch  1/10


  train:   7%|████▏                                                     | 51/704 [00:30<06:08,  1.77it/s]

    batch   50/704 | loss 1.00960


  train:  14%|████████▎                                                | 102/704 [00:57<04:34,  2.19it/s]

    batch  100/704 | loss 0.81639


...

  train:  93%|████████████████████████████████████████████████████▊    | 652/704 [05:36<00:32,  1.59it/s]

    batch  650/704 | loss 0.34329


  train:  99%|████████████████████████████████████████████████████████▌| 699/704 [05:58<00:01,  2.70it/s]

    batch  700/704 | loss 0.34238


  train loss : 0.34244
  val acc    : 83.56%
  val AUC    : 0.93706
  lr         : 8.32e-06
  ✓ New best AUC 0.93706 — saved to /workspace/dinov2_mlp_best.pth

Training complete.  Best val AUC: 0.93706

Training history saved to: /workspace/dinov2_mlp_training_history.csv
 epoch  train_loss  val_acc  val_auc       lr
     1    0.559940   0.7954 0.892317 0.000300
     2    0.452877   0.8086 0.910063 0.000293
     3    0.416396   0.8252 0.919893 0.000271
     4    0.393411   0.8336 0.925003 0.000238
     5    0.379709   0.8274 0.929024 0.000197
     6    0.364708   0.8336 0.932444 0.000150
     7    0.357463   0.8382 0.934456 0.000104
     8    0.351936   0.8370 0.936722 0.000063
     9    0.351100   0.8362 0.936708 0.000030
    10    0.342436   0.8356 0.937057 0.000008


## 13. Save Final Model
Save the final-epoch weights alongside the already-saved best checkpoint.


In [13]:
torch.save(model.state_dict(), FINAL_PATH)
print(f"Final model saved : {FINAL_PATH}")
print(f"Best  model saved : {BEST_PATH}  (AUC {best_auc:.5f})")


Final model saved : /workspace/dinov2_mlp_final.pth
Best  model saved : /workspace/dinov2_mlp_best.pth  (AUC 0.93706)


## 14. Multi-Crop Inference (5-Crop)
`torchvision.transforms.FiveCrop(224)` produces five 224×224 crops from a 256×256 image: the four corners plus the centre.  
Running the model on all five crops and averaging their sigmoid scores reduces spatial bias and typically improves AUC by 0.5–1 point over single-crop evaluation.

Each image is processed independently — no dataloader needed, just one pass per crop stack.


In [14]:
from torchvision.transforms import FiveCrop

# ── 5-crop pre-processing pipeline ───────────────────────────────────────────
# Resize to 256 first so each of the 5 crops has the full 224 content budget.
_five_crop_pre = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    FiveCrop(CFG["img_size"]),           # returns a tuple of 5 PIL images
])
_to_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=_MEAN, std=_STD),
])


@torch.no_grad()
def five_crop_score(pil_img: Image.Image) -> float:
    """
    Run 5-crop evaluation on a single PIL image.

    Steps
    -----
    1. Resize to 256, then take 5 crops (4 corners + centre) of size 224.
    2. Stack the 5 crops into a (5, 3, 224, 224) batch.
    3. Forward through the best-checkpoint model.
    4. Average sigmoid(logit) across the 5 crops.

    Returns
    -------
    score : float in [0, 1] — probability of AI-generated
    """
    crops      = _five_crop_pre(pil_img)                        # tuple of 5 PIL images
    crop_batch = torch.stack([_to_tensor(c) for c in crops])    # (5, 3, 224, 224)
    crop_batch = crop_batch.to(device)

    with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
        logits = inference_model(crop_batch)                    # (5, 1)

    return torch.sigmoid(logits).mean().item()                  # scalar in [0, 1]


# ── Load the best checkpoint for inference ────────────────────────────────────
inference_model = DINOv2Classifier(backbone, embed_dim=EMBED_DIM, dropout=0.0).to(device)
inference_model.load_state_dict(torch.load(BEST_PATH, map_location=device))
inference_model.eval()
print(f"Loaded best model from: {BEST_PATH}")

# ── Run 5-crop inference over the validation split ────────────────────────────
# val_dataset already contains only the validation rows (indices were applied at
# construction time), so we iterate its .labels DataFrame directly.
all_scores:      list[float] = []
all_preds:       list[int]   = []
all_true_labels: list[int]   = []
all_image_names: list[str]   = []

for i in tqdm(range(len(val_dataset)), desc="5-crop inference"):
    row      = val_dataset.labels.iloc[i]
    img_path = os.path.join(CFG["dataset_path"], "images", row["image_name"])
    pil_img  = Image.open(img_path).convert("RGB")

    score = five_crop_score(pil_img)
    all_scores.append(score)
    all_preds.append(int(score >= 0.5))
    all_true_labels.append(int(row["label"]))
    all_image_names.append(row["image_name"])

# ── Build results DataFrame ───────────────────────────────────────────────────
fivecrop_df = pd.DataFrame({
    "image_name": all_image_names,
    "score":      all_scores,          # probability of AI-generated, in [0, 1]
    "predicted":  all_preds,
    "true_label": all_true_labels,
    "correct":    [p == t for p, t in zip(all_preds, all_true_labels)],
})
fivecrop_df = fivecrop_df.sort_values("score", ascending=False).reset_index(drop=True)

# ── Metrics ───────────────────────────────────────────────────────────────────
fc_acc = fivecrop_df["correct"].mean()
fc_auc = roc_auc_score(fivecrop_df["true_label"], fivecrop_df["score"])

out_csv = os.path.join(CFG["ckpt_dir"], "dinov2_mlp_fivecrop_predictions.csv")
fivecrop_df.to_csv(out_csv, index=False)

print(f"\n5-Crop inference complete — {len(fivecrop_df)} samples")
print(f"Accuracy : {fc_acc * 100:.2f}%")
print(f"ROC-AUC  : {fc_auc:.5f}")
print(f"Results saved to: {out_csv}")
print("\nTop-10 highest-confidence AI detections:")
print(fivecrop_df.head(10).to_string(index=False))


Loaded best model from: /workspace/dinov2_mlp_best.pth


5-crop inference: 100%|██████████████████████████████████████████████| 5000/5000 [07:13<00:00, 11.54it/s]


5-Crop inference complete — 5000 samples
Accuracy : 84.82%
ROC-AUC  : 0.94727
Results saved to: /workspace/dinov2_mlp_fivecrop_predictions.csv

Top-10 highest-confidence AI detections:
              image_name    score  predicted  true_label  correct
678c0827347f1957cf69.jpg 1.000000          1           1     True
18a56ac82f5f61fe1a8d.jpg 1.000000          1           1     True
5f4a80f16634ab632172.jpg 1.000000          1           1     True
5f07604585ade23cab4e.jpg 1.000000          1           1     True
86915e57b8aadbe4fbdb.jpg 1.000000          1           1     True
5aea9981933a6414f35a.jpg 1.000000          1           1     True
07c1b13d2c365cbc7b69.jpg 1.000000          1           1     True
01cb755a69d22ef6aa36.jpg 1.000000          1           1     True
4a1acffe0ca2a0d3ba7c.jpg 0.999512          1           1     True
e40f9ee494a807b2116e.jpg 0.999512          1           1     True
